In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Cargar los 4 datasets RENAMU
df_original   = pd.read_csv('Base_RENAMU_2022_f.csv', encoding='latin-1', sep=';')
df_simple     = pd.read_parquet('renamu_silver_simple.parquet')
df_intermedio = pd.read_parquet('renamu_silver_intermedio.parquet')
df_avanzado   = pd.read_parquet('renamu_silver_avanzado.parquet')

datasets = {
    "Original":   df_original,
    "Simple":     df_simple,
    "Intermedio": df_intermedio,
    "Avanzado":   df_avanzado
}

print("✅ Datasets RENAMU cargados correctamente")
for nombre, df in datasets.items():
    print(f"   {nombre:<12}: {len(df):,} registros, {len(df.columns)} columnas")

In [ ]:
# Eliminar columnas temporales generadas durante la limpieza
columnas_temp = ['Ubigeo_calculado']

for nombre, df_temp in datasets.items():
    for col in columnas_temp:
        if col in df_temp.columns:
            df_temp.drop(columns=[col], inplace=True)

print("✅ Columnas temporales eliminadas")

In [ ]:
# Clave de negocio para duplicados RENAMU
cols_duplicados = ['Ubigeo']

def get_quality_metrics(df):
    existing_cols = [col for col in cols_duplicados if col in df.columns]

    ubigeo_inv = len(df[
        ~df['Ubigeo'].astype(str).str.match(r'^\d{6}$')
    ]) if 'Ubigeo' in df.columns else 0

    tipomuni_inv = len(df[
        ~df['Tipomuni'].isin([1, 2, 3, 4, 5])
    ]) if 'Tipomuni' in df.columns else 0

    return {
        "registros":      len(df),
        "nulos":          df[['Ubigeo','Departamento','Provincia','Distrito']].isnull().sum().sum()
                          if all(c in df.columns for c in ['Ubigeo','Departamento','Provincia','Distrito'])
                          else df.isnull().sum().sum(),
        "duplicados":     df.duplicated(subset=existing_cols).sum() if existing_cols else 0,
        "ubigeo_invalido": ubigeo_inv,
        "tipomuni_invalido": tipomuni_inv
    }

In [ ]:
base_size  = len(df_original)
comparison = []

for nombre, df_temp in datasets.items():
    metrics = get_quality_metrics(df_temp)

    comparison.append({
        "Dataset":            nombre,
        "Registros":          metrics["registros"],
        "Pérdida (%)":        round((base_size - metrics["registros"]) / base_size * 100, 2),
        "Nulos críticos":     metrics["nulos"],
        "Duplicados":         metrics["duplicados"],
        "Ubigeo inválido":    metrics["ubigeo_invalido"],
        "Tipomuni inválido":  metrics["tipomuni_invalido"]
    })

comparison_df = pd.DataFrame(comparison)

print("=" * 70)
print("   COMPARATIVA DE CALIDAD — RENAMU 2022")
print("=" * 70)
display(comparison_df)

In [ ]:
colores = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

# Gráfico 1: Registros por dataset
plt.figure(figsize=(10, 5))
bars = plt.bar(comparison_df["Dataset"], comparison_df["Registros"],
               color=colores, edgecolor='white')
plt.title('Registros por Dataset\nRENAMU 2022', fontsize=13, fontweight='bold')
plt.ylabel("Cantidad de registros")
plt.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, comparison_df["Registros"]):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 1,
             f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Pérdida de datos por dataset
plt.figure(figsize=(10, 5))
bars = plt.bar(comparison_df["Dataset"], comparison_df["Pérdida (%)"],
               color=colores, edgecolor='white')
plt.title('Pérdida de Datos (%)\nRENAMU 2022', fontsize=13, fontweight='bold')
plt.ylabel("% pérdida respecto al dataset original")
plt.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, comparison_df["Pérdida (%)"]):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.1,
             f'{v}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: Problemas de calidad por dataset
problemas_cols = [
    "Nulos críticos",
    "Duplicados",
    "Ubigeo inválido",
    "Tipomuni inválido"
]

x     = np.arange(len(problemas_cols))
width = 0.2

plt.figure(figsize=(13, 6))

for i, (nombre, df_temp) in enumerate(datasets.items()):
    metrics = get_quality_metrics(df_temp)
    values  = [
        metrics["nulos"],
        metrics["duplicados"],
        metrics["ubigeo_invalido"],
        metrics["tipomuni_invalido"]
    ]
    bars = plt.bar(x + i * width, values, width,
                   label=nombre, color=colores[i], edgecolor='white')
    for bar, v in zip(bars, values):
        if v > 0:
            plt.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.5,
                     f'{v:,}', ha='center', va='bottom', fontsize=7)

plt.xticks(x + width * 1.5, problemas_cols, rotation=15, ha='right')
plt.title('Problemas de Calidad por Dataset\nRENAMU 2022',
          fontsize=13, fontweight='bold')
plt.ylabel("Cantidad de registros con problema")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def quality_score(metrics, total_rows):
    issues = (
        metrics["nulos"]            +
        metrics["duplicados"]       +
        metrics["ubigeo_invalido"]  +
        metrics["tipomuni_invalido"]
    )
    score = max(0, 100 - (issues / total_rows * 100)) if total_rows > 0 else 0
    return round(score, 2)


scores = []

for nombre, df_temp in datasets.items():
    m     = get_quality_metrics(df_temp)
    score = quality_score(m, m["registros"])
    scores.append(score)

colores_score = [
    '#e74c3c' if s < 70 else '#f39c12' if s < 90 else '#2ecc71'
    for s in scores
]

plt.figure(figsize=(10, 5))
bars = plt.bar(datasets.keys(), scores,
               color=colores_score, edgecolor='white')
plt.axhline(y=90, color='#2ecc71', linestyle='--',
            linewidth=1.5, label='Meta calidad (90)')
plt.axhline(y=70, color='#e74c3c', linestyle='--',
            linewidth=1.5, label='Umbral mínimo (70)')
plt.title('Data Quality Score por Dataset\nRENAMU 2022',
          fontsize=13, fontweight='bold')
plt.ylabel("Score (0-100)")
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3)
plt.legend(fontsize=9)
for bar, v in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5,
             f'{v}', ha='center', va='bottom',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Quality Scores RENAMU 2022:")
for nombre, score in zip(datasets.keys(), scores):
    estado = '🟢 BUENO' if score >= 90 else '🟡 REGULAR' if score >= 70 else '🔴 BAJO'
    print(f"   {nombre:<12}: {score:>6} — {estado}")